In [1]:
import itertools
import os
import random

import joblib
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC, LinearSVC

SEED = 42
TARGET = 0.90
NS = (20, 30, 40)
FRACS = (0.25, 0.5, 1.0)
SVM_MAX_ITER = 10000  # iteration cap for every SVM fit
WEIGHT_ROOT = "./weights"

random.seed(SEED)
np.random.seed(SEED)

## Data

In [2]:
def load(name):
    """Load (X, y) arrays for a dataset."""
    return np.load(f"Datasets/{name}-X.npy"), np.load(f"Datasets/{name}-y.npy")


def split(X, y):
    """60% train, 20% validation, 20% test."""
    X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.4, random_state=SEED)
    X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=SEED)
    return X_train, y_train, X_val, y_val, X_test, y_test


def binarise(X):
    """Map each noisy +-1 feature to its sign."""
    return np.where(X > 0, 1.0, -1.0)

## Feature selection

In [3]:
def lookup_table_acc(S_train, y_train, S_val, y_val, cols):
    """Val accuracy of a majority-vote lookup table over the bits in `cols`."""
    w = 2 ** np.arange(len(cols))
    idx_tr = ((S_train[:, cols] > 0) * w).sum(1)
    idx_va = ((S_val[:, cols] > 0) * w).sum(1)
    count = np.bincount(idx_tr, minlength=2 ** len(cols))
    pos = np.bincount(idx_tr, weights=y_train, minlength=2 ** len(cols))
    table = (2 * pos > count).astype(int)  # unseen cells default to 0
    return (table[idx_va] == y_val).mean()


def select_features(S_train, y_train, S_val, y_val, min_gain=0.005):
    """Greedy forward selection that may add one bit or a pair of bits per step."""
    n = S_train.shape[1]
    score = lambda cols: lookup_table_acc(S_train, y_train, S_val, y_val, list(cols))
    selected, best = [], 0.5
    while True:
        rest = [j for j in range(n) if j not in selected]
        cands = [[j] for j in rest] + [list(p) for p in itertools.combinations(rest, 2)]
        acc, add = max((score(selected + c), c) for c in cands)
        if acc <= best + min_gain:
            break
        selected, best = selected + add, acc
        print(f"  + {add} -> {sorted(selected)}  val acc {acc:.4f}")
    if not selected:
        print("  no feature set beats chance; keeping all features")
        return list(range(n))
    return sorted(selected)

## Models

In [4]:
def tune(name, make, grid, train, val):
    """Fit `make(**params)` for every params in `grid`; keep the best on val (first wins ties)."""
    (Xtr, ytr), (Xva, yva) = train, val
    best = None
    for params in grid:
        model = make(**params).fit(Xtr, ytr)
        acc = model.score(Xva, yva)
        if best is None or acc > best[0]:
            best = (acc, params, model)
    print(f"  {name}: best {best[1]}  val acc {best[0]:.4f}")
    return best


linear_grid = [dict(C=C) for C in [0.001, 0.01, 0.1, 1.0, 10.0, 100.0]]
poly_grid = [dict(C=C, degree=d, coef0=c0)
             for C in [0.1, 1.0, 10.0] for d in [2, 3, 4] for c0 in [0.0, 1.0]]
rbf_grid = [dict(C=C, gamma=g)
            for C in [0.1, 1.0, 10.0, 100.0] for g in ["scale", 0.01, 0.1, 0.25, 0.5, 1.0]]

# (display name, weight folder, constructor, grid)
model_specs = [
    ("Linear SVM", "linear_svm",
     lambda **p: LinearSVC(dual=False, max_iter=SVM_MAX_ITER, random_state=SEED, **p), linear_grid),
    ("Polynomial SVM", "poly_svm",
     lambda **p: SVC(kernel="poly", gamma="scale", cache_size=512, max_iter=SVM_MAX_ITER,
                     random_state=SEED, **p), poly_grid),
    ("RBF SVM", "rbf_svm",
     lambda **p: SVC(kernel="rbf", cache_size=512, max_iter=SVM_MAX_ITER, random_state=SEED, **p), rbf_grid),
]
W = max(len(name) for name, *_ in model_specs)

## All datasets and scales

In [5]:
def run(n, frac):
    """Feature selection on the first `frac` of train/val, tune every model, save each best model."""
    X_train, y_train, X_val, y_val, X_test, y_test = split(*load(f"kryptonite-{n}"))
    m, mv = int(frac * len(y_train)), int(frac * len(y_val))
    S_tr, y_tr = binarise(X_train[:m]), y_train[:m]
    S_va, y_va = binarise(X_val[:mv]), y_val[:mv]
    S_te, y_te = binarise(X_test), y_test
    print(f"\n=== n={n}, {frac:.0%}: {m} train / {mv} val / {len(y_te)} test ===")

    cols = select_features(S_tr, y_tr, S_va, y_va)
    print(f"  selected {len(cols)} features: {cols}")
    F_tr, F_va, F_te = S_tr[:, cols], S_va[:, cols], S_te[:, cols]

    rows = []
    for model_name, folder, make, grid in model_specs:
        val_acc, params, model = tune(model_name, make, grid, (F_tr, y_tr), (F_va, y_va))
        test_acc = model.score(F_te, y_te)
        out_dir = os.path.join(WEIGHT_ROOT, folder, f"n{n}", f"{round(frac * 100)}pct")
        os.makedirs(out_dir, exist_ok=True)
        path = os.path.join(out_dir, "model.joblib")
        joblib.dump({"model": model, "features": cols, "params": params}, path)
        print(f"  {model_name:<{W}s} val {val_acc:.4f}  test {test_acc:.4f}  -> {path}")
        rows.append({"n": n, "frac": frac, "model": model_name, "train_size": m, "features": cols,
                     "params": params, "val": val_acc, "test": test_acc, "path": path})
    return rows


results = [row for n in NS for frac in FRACS for row in run(n, frac)]


=== n=20, 25%: 1800 train / 600 val / 2400 test ===
  + [4, 9] -> [4, 9]  val acc 0.5550
  + [10, 12] -> [4, 9, 10, 12]  val acc 0.6117
  + [5, 15] -> [4, 5, 9, 10, 12, 15]  val acc 0.6583
  + [1, 7] -> [1, 4, 5, 7, 9, 10, 12, 15]  val acc 0.7350
  + [0, 2] -> [0, 1, 2, 4, 5, 7, 9, 10, 12, 15]  val acc 0.8700
  selected 10 features: [0, 1, 2, 4, 5, 7, 9, 10, 12, 15]
  Linear SVM: best {'C': 0.01}  val acc 0.5150
  Linear SVM     val 0.5150  test 0.5142  -> ./weights\linear_svm\n20\25pct\model.joblib


c:\Users\19364\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\svm\_base.py:337: ConvergenceWarning: Solver terminated early (max_iter=10000).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(
c:\Users\19364\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\svm\_base.py:337: ConvergenceWarning: Solver terminated early (max_iter=10000).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(
c:\Users\19364\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\svm\_base.py:337: ConvergenceWarning: Solver terminated early (max_iter=10000).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(
c:\Users\19364\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\svm\_base.py:337: ConvergenceWarning: Solver terminated early (max_iter=10000).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(


  Polynomial SVM: best {'C': 1.0, 'degree': 4, 'coef0': 1.0}  val acc 0.6567
  Polynomial SVM val 0.6567  test 0.6775  -> ./weights\poly_svm\n20\25pct\model.joblib


c:\Users\19364\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\svm\_base.py:337: ConvergenceWarning: Solver terminated early (max_iter=10000).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(
c:\Users\19364\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\svm\_base.py:337: ConvergenceWarning: Solver terminated early (max_iter=10000).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(
c:\Users\19364\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\svm\_base.py:337: ConvergenceWarning: Solver terminated early (max_iter=10000).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(
c:\Users\19364\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\svm\_base.py:337: ConvergenceWarning: Solver terminated early (max_iter=10000).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(


  RBF SVM: best {'C': 1.0, 'gamma': 1.0}  val acc 0.8283
  RBF SVM        val 0.8283  test 0.8358  -> ./weights\rbf_svm\n20\25pct\model.joblib

=== n=20, 50%: 3600 train / 1200 val / 2400 test ===
  + [4, 9] -> [4, 9]  val acc 0.5750
  + [10, 12] -> [4, 9, 10, 12]  val acc 0.6283
  + [5, 15] -> [4, 5, 9, 10, 12, 15]  val acc 0.6842
  + [1, 7] -> [1, 4, 5, 7, 9, 10, 12, 15]  val acc 0.7692
  + [0, 2] -> [0, 1, 2, 4, 5, 7, 9, 10, 12, 15]  val acc 0.9400
  selected 10 features: [0, 1, 2, 4, 5, 7, 9, 10, 12, 15]
  Linear SVM: best {'C': 0.01}  val acc 0.4808
  Linear SVM     val 0.4808  test 0.5038  -> ./weights\linear_svm\n20\50pct\model.joblib


c:\Users\19364\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\svm\_base.py:337: ConvergenceWarning: Solver terminated early (max_iter=10000).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(
c:\Users\19364\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\svm\_base.py:337: ConvergenceWarning: Solver terminated early (max_iter=10000).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(
c:\Users\19364\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\svm\_base.py:337: ConvergenceWarning: Solver terminated early (max_iter=10000).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(
c:\Users\19364\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\svm\_base.py:337: ConvergenceWarning: Solver terminated early (max_iter=10000).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(


... (27 outputs left out)

c:\Users\19364\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\svm\_base.py:337: ConvergenceWarning: Solver terminated early (max_iter=10000).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(
c:\Users\19364\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\svm\_base.py:337: ConvergenceWarning: Solver terminated early (max_iter=10000).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(
c:\Users\19364\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\svm\_base.py:337: ConvergenceWarning: Solver terminated early (max_iter=10000).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(
c:\Users\19364\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\svm\_base.py:337: ConvergenceWarning: Solver terminated early (max_iter=10000).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(


  Polynomial SVM: best {'C': 0.1, 'degree': 4, 'coef0': 1.0}  val acc 0.6837
  Polynomial SVM val 0.6837  test 0.6872  -> ./weights\poly_svm\n40\100pct\model.joblib


c:\Users\19364\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\svm\_base.py:337: ConvergenceWarning: Solver terminated early (max_iter=10000).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(
c:\Users\19364\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\svm\_base.py:337: ConvergenceWarning: Solver terminated early (max_iter=10000).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(
c:\Users\19364\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\svm\_base.py:337: ConvergenceWarning: Solver terminated early (max_iter=10000).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(
c:\Users\19364\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\svm\_base.py:337: ConvergenceWarning: Solver terminated early (max_iter=10000).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(


  RBF SVM: best {'C': 0.1, 'gamma': 1.0}  val acc 0.9569
  RBF SVM        val 0.9569  test 0.9631  -> ./weights\rbf_svm\n40\100pct\model.joblib


## Summary

In [6]:
print(f"{'n':>3} | {'subset':>6} | {'train':>6} | {'model':<{W}s} | {'#feat':>5} | {'val':>6} | {'test':>6}")
for r in results:
    flag = "" if r["test"] >= TARGET else "  < target"
    print(f"{r['n']:>3} | {r['frac']:>6.0%} | {r['train_size']:>6} | {r['model']:<{W}s} | "
          f"{len(r['features']):>5} | {r['val']:.4f} | {r['test']:.4f}{flag}")

  n | subset |  train | model          | #feat |    val |   test
 20 |    25% |   1800 | Linear SVM     |    10 | 0.5150 | 0.5142  < target
 20 |    25% |   1800 | Polynomial SVM |    10 | 0.6567 | 0.6775  < target
 20 |    25% |   1800 | RBF SVM        |    10 | 0.8283 | 0.8358  < target
 20 |    50% |   3600 | Linear SVM     |    10 | 0.4808 | 0.5038  < target
 20 |    50% |   3600 | Polynomial SVM |    10 | 0.6975 | 0.7037  < target
 20 |    50% |   3600 | RBF SVM        |    10 | 0.9367 | 0.9400
 20 |   100% |   7200 | Linear SVM     |    10 | 0.5096 | 0.5058  < target
 20 |   100% |   7200 | Polynomial SVM |    10 | 0.7129 | 0.7275  < target
 20 |   100% |   7200 | RBF SVM        |    10 | 0.9533 | 0.9592
 30 |    25% |   3000 | Linear SVM     |    10 | 0.4610 | 0.4873  < target
 30 |    25% |   3000 | Polynomial SVM |    10 | 0.6740 | 0.6865  < target
 30 |    25% |   3000 | RBF SVM        |    10 | 0.9050 | 0.9100
 30 |    50% |   6000 | Linear SVM     |    10 | 0.4740 | 0.5085 

Reload every saved model and check it reproduces its test accuracy.

In [7]:
for r in results:
    saved = joblib.load(r["path"])
    _, _, _, _, X_test, y_test = split(*load(f"kryptonite-{r['n']}"))
    acc = saved["model"].score(binarise(X_test)[:, saved["features"]], y_test)
    assert abs(acc - r["test"]) < 1e-9, (r["path"], acc, r["test"])
    print(f"{r['path']} -> test {acc:.4f}")

./weights\linear_svm\n20\25pct\model.joblib -> test 0.5142
./weights\poly_svm\n20\25pct\model.joblib -> test 0.6775
./weights\rbf_svm\n20\25pct\model.joblib -> test 0.8358
./weights\linear_svm\n20\50pct\model.joblib -> test 0.5038
./weights\poly_svm\n20\50pct\model.joblib -> test 0.7037
./weights\rbf_svm\n20\50pct\model.joblib -> test 0.9400
./weights\linear_svm\n20\100pct\model.joblib -> test 0.5058
./weights\poly_svm\n20\100pct\model.joblib -> test 0.7275
./weights\rbf_svm\n20\100pct\model.joblib -> test 0.9592
./weights\linear_svm\n30\25pct\model.joblib -> test 0.4873
./weights\poly_svm\n30\25pct\model.joblib -> test 0.6865
./weights\rbf_svm\n30\25pct\model.joblib -> test 0.9100
./weights\linear_svm\n30\50pct\model.joblib -> test 0.5085
./weights\poly_svm\n30\50pct\model.joblib -> test 0.7000
./weights\rbf_svm\n30\50pct\model.joblib -> test 0.9543
./weights\linear_svm\n30\100pct\model.joblib -> test 0.5242
./weights\poly_svm\n30\100pct\model.joblib -> test 0.6957
./weights\rbf_svm\n

## No feature selection (ablation)

The same models, grids, split and data fractions as above, but every model is tuned on all n sign features instead of the selected ones. The best model per (n, fraction) is saved to `weights/<model>_no_fs/n<n>/<pct>pct/model.joblib` in the same layout, with `features` set to all n columns.

In [8]:
def run_no_fs(n, frac):
    """Use the first `frac` of train/val with all n features, tune every model, save each best model."""
    X_train, y_train, X_val, y_val, X_test, y_test = split(*load(f"kryptonite-{n}"))
    m, mv = int(frac * len(y_train)), int(frac * len(y_val))
    S_tr, y_tr = binarise(X_train[:m]), y_train[:m]
    S_va, y_va = binarise(X_val[:mv]), y_val[:mv]
    S_te, y_te = binarise(X_test), y_test
    print(f"\n=== n={n}, {frac:.0%}, no FS: {m} train / {mv} val / {len(y_te)} test ===")

    cols = list(range(n))  # no feature selection: all features are used
    width = max(len(name) for name, *_ in model_specs)
    rows = []
    for model_name, folder, make, grid in model_specs:
        val_acc, params, model = tune(model_name, make, grid, (S_tr, y_tr), (S_va, y_va))
        test_acc = model.score(S_te, y_te)
        out_dir = os.path.join(WEIGHT_ROOT, f"{folder}_no_fs", f"n{n}", f"{round(frac * 100)}pct")
        os.makedirs(out_dir, exist_ok=True)
        path = os.path.join(out_dir, "model.joblib")
        joblib.dump({"model": model, "features": cols, "params": params}, path)
        print(f"  {model_name:<{width}s} val {val_acc:.4f}  test {test_acc:.4f}  -> {path}")
        rows.append({"n": n, "frac": frac, "model": model_name, "train_size": m, "features": cols,
                     "params": params, "val": val_acc, "test": test_acc, "path": path})
    return rows


results_no_fs = [row for n in NS for frac in FRACS for row in run_no_fs(n, frac)]


=== n=20, 25%, no FS: 1800 train / 600 val / 2400 test ===
  Linear SVM: best {'C': 0.01}  val acc 0.4933
  Linear SVM     val 0.4933  test 0.5092  -> ./weights\linear_svm_no_fs\n20\25pct\model.joblib


c:\Users\19364\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\svm\_base.py:337: ConvergenceWarning: Solver terminated early (max_iter=10000).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(
c:\Users\19364\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\svm\_base.py:337: ConvergenceWarning: Solver terminated early (max_iter=10000).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(
c:\Users\19364\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\svm\_base.py:337: ConvergenceWarning: Solver terminated early (max_iter=10000).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(
c:\Users\19364\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\svm\_base.py:337: ConvergenceWarning: Solver terminated early (max_iter=10000).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(


  Polynomial SVM: best {'C': 10.0, 'degree': 4, 'coef0': 0.0}  val acc 0.5500
  Polynomial SVM val 0.5500  test 0.5333  -> ./weights\poly_svm_no_fs\n20\25pct\model.joblib


c:\Users\19364\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\svm\_base.py:337: ConvergenceWarning: Solver terminated early (max_iter=10000).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(
c:\Users\19364\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\svm\_base.py:337: ConvergenceWarning: Solver terminated early (max_iter=10000).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(
c:\Users\19364\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\svm\_base.py:337: ConvergenceWarning: Solver terminated early (max_iter=10000).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(


  RBF SVM: best {'C': 10.0, 'gamma': 0.25}  val acc 0.5500
  RBF SVM        val 0.5500  test 0.5458  -> ./weights\rbf_svm_no_fs\n20\25pct\model.joblib

=== n=20, 50%, no FS: 3600 train / 1200 val / 2400 test ===
  Linear SVM: best {'C': 0.001}  val acc 0.4892
  Linear SVM     val 0.4892  test 0.5125  -> ./weights\linear_svm_no_fs\n20\50pct\model.joblib


c:\Users\19364\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\svm\_base.py:337: ConvergenceWarning: Solver terminated early (max_iter=10000).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(
c:\Users\19364\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\svm\_base.py:337: ConvergenceWarning: Solver terminated early (max_iter=10000).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(
c:\Users\19364\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\svm\_base.py:337: ConvergenceWarning: Solver terminated early (max_iter=10000).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(
c:\Users\19364\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\svm\_base.py:337: ConvergenceWarning: Solver terminated early (max_iter=10000).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(


... (27 outputs left out)

c:\Users\19364\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\svm\_base.py:337: ConvergenceWarning: Solver terminated early (max_iter=10000).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(
c:\Users\19364\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\svm\_base.py:337: ConvergenceWarning: Solver terminated early (max_iter=10000).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(
c:\Users\19364\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\svm\_base.py:337: ConvergenceWarning: Solver terminated early (max_iter=10000).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(
c:\Users\19364\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\svm\_base.py:337: ConvergenceWarning: Solver terminated early (max_iter=10000).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(


  Polynomial SVM: best {'C': 0.1, 'degree': 3, 'coef0': 1.0}  val acc 0.5473
  Polynomial SVM val 0.5473  test 0.5514  -> ./weights\poly_svm_no_fs\n40\100pct\model.joblib


c:\Users\19364\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\svm\_base.py:337: ConvergenceWarning: Solver terminated early (max_iter=10000).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(
c:\Users\19364\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\svm\_base.py:337: ConvergenceWarning: Solver terminated early (max_iter=10000).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(
c:\Users\19364\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\svm\_base.py:337: ConvergenceWarning: Solver terminated early (max_iter=10000).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(
c:\Users\19364\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\svm\_base.py:337: ConvergenceWarning: Solver terminated early (max_iter=10000).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(


  RBF SVM: best {'C': 1.0, 'gamma': 'scale'}  val acc 0.5516
  RBF SVM        val 0.5516  test 0.5480  -> ./weights\rbf_svm_no_fs\n40\100pct\model.joblib


### Summary without feature selection

In [9]:
width = max(len(name) for name, *_ in model_specs)
print(f"{'n':>3} | {'subset':>6} | {'train':>6} | {'model':<{width}s} | {'#feat':>5} | {'val':>6} | {'test':>6}")
for r in results_no_fs:
    flag = "" if r["test"] >= TARGET else "  < target"
    print(f"{r['n']:>3} | {r['frac']:>6.0%} | {r['train_size']:>6} | {r['model']:<{width}s} | "
          f"{len(r['features']):>5} | {r['val']:.4f} | {r['test']:.4f}{flag}")

  n | subset |  train | model          | #feat |    val |   test
 20 |    25% |   1800 | Linear SVM     |    20 | 0.4933 | 0.5092  < target
 20 |    25% |   1800 | Polynomial SVM |    20 | 0.5500 | 0.5333  < target
 20 |    25% |   1800 | RBF SVM        |    20 | 0.5500 | 0.5458  < target
 20 |    50% |   3600 | Linear SVM     |    20 | 0.4892 | 0.5125  < target
 20 |    50% |   3600 | Polynomial SVM |    20 | 0.5683 | 0.5254  < target
 20 |    50% |   3600 | RBF SVM        |    20 | 0.5850 | 0.5713  < target
 20 |   100% |   7200 | Linear SVM     |    20 | 0.5000 | 0.5033  < target
 20 |   100% |   7200 | Polynomial SVM |    20 | 0.5642 | 0.5654  < target
 20 |   100% |   7200 | RBF SVM        |    20 | 0.5754 | 0.5938  < target
 30 |    25% |   3000 | Linear SVM     |    30 | 0.4860 | 0.5028  < target
 30 |    25% |   3000 | Polynomial SVM |    30 | 0.5610 | 0.5115  < target
 30 |    25% |   3000 | RBF SVM        |    30 | 0.5710 | 0.5150  < target
 30 |    50% |   6000 | Linear SVM 

Reload every saved no-FS model and check it reproduces its test accuracy.

In [10]:
for r in results_no_fs:
    saved = joblib.load(r["path"])
    _, _, _, _, X_test, y_test = split(*load(f"kryptonite-{r['n']}"))
    acc = saved["model"].score(binarise(X_test)[:, saved["features"]], y_test)
    assert abs(acc - r["test"]) < 1e-9, (r["path"], acc, r["test"])
    print(f"{r['path']} -> test {acc:.4f}")

./weights\linear_svm_no_fs\n20\25pct\model.joblib -> test 0.5092
./weights\poly_svm_no_fs\n20\25pct\model.joblib -> test 0.5333
./weights\rbf_svm_no_fs\n20\25pct\model.joblib -> test 0.5458
./weights\linear_svm_no_fs\n20\50pct\model.joblib -> test 0.5125
./weights\poly_svm_no_fs\n20\50pct\model.joblib -> test 0.5254
./weights\rbf_svm_no_fs\n20\50pct\model.joblib -> test 0.5713
./weights\linear_svm_no_fs\n20\100pct\model.joblib -> test 0.5033
./weights\poly_svm_no_fs\n20\100pct\model.joblib -> test 0.5654
./weights\rbf_svm_no_fs\n20\100pct\model.joblib -> test 0.5938
./weights\linear_svm_no_fs\n30\25pct\model.joblib -> test 0.5028
./weights\poly_svm_no_fs\n30\25pct\model.joblib -> test 0.5115
./weights\rbf_svm_no_fs\n30\25pct\model.joblib -> test 0.5150
./weights\linear_svm_no_fs\n30\50pct\model.joblib -> test 0.5062
./weights\poly_svm_no_fs\n30\50pct\model.joblib -> test 0.5417
./weights\rbf_svm_no_fs\n30\50pct\model.joblib -> test 0.5380
./weights\linear_svm_no_fs\n30\100pct\model.job